# Сводка результатов всех экспериментов (experiments/)

Этот ноутбук собирает `result.csv` из всех экспериментов в единую таблицу  
и строит визуализации для сравнения.

## Структура папки experiments/

```
01_baselines/            # Классический ML
  exp_mfcc_svm/          SVM на MFCC stats
  exp_vggish_svm/        SVM на VGGish эмбеддингах

02_spectrogram_models/   # DL на спектрограммах
  exp_dsscnet/           DSSCNet (SE+Residual) + SpecAugment
  exp_bilstm_gru/        BiLSTM-GRU + SpecAugment
  exp_fluentnet/         FluentNet (1D CNN + 2D CNN) + SpecAugment
  exp_cross_attn/        CNN-BiLSTM + Cross-Attention + SpecAugment

03_pretrained_frozen/    # Замороженные энкодеры
  exp_whisper_svm/       Whisper-small + SVM
  exp_whisper_medium_svm/ Whisper-medium + SVM
  exp_whisper_large_svm/ Whisper-large + SVM
  exp_whisper_dysarthria_svm/ Whisper (Russian dysarthria) + SVM
  exp_wav2vec_svm/       Wav2Vec2-base + SVM
  exp_xlsr_svm/          XLS-R-300M + SVM
  exp_hubert_lstm/       HuBERT + BiLSTM (предвычисленные эмбеддинги)
  exp_openl3_svm/        OpenL3 + SVM

04_pretrained_finetuned/ # Fine-tuning энкодеров
  exp_whisper_finetune/  Whisper partial finetune

05_ensembles/            # Ансамбли
  exp_soft_voting/       Soft voting: Whisper + OpenL3 + XLS-R

06_phoneme_analysis/     # Фонемный анализ
  exp_whisper_phoneme/   Per-phoneme метрики Whisper+SVM
```

**Итого: 17 экспериментов** (F1-bad ≥ 0.60 из checkpoint_3 + новые группы 04–06)

## Исправления по сравнению с checkpoint_3

| Проблема | Решение |
|----------|----------|
| Один сплит 70/15/15 → selection bias | Holdout test изолируется первым, CV только на train+val |
| Порог 0.5 везде | Оптимизация порога по F1-bad на val в каждом эксперименте |
| Нет аугментации | SpecAugment в DL-экспериментах (группа 02) |
| HuBERT encoder запускался каждую эпоху (9298 с) | Эмбеддинги предвычисляются один раз |
| Wav2Vec2 DL head занимал 9566 с | Переход на SVM на предвычисленных эмбеддингах |
| Буквы управления только как признаки | Отдельный фонемный анализ (группа 06) |
| Нет ансамблей | Soft voting лучших трёх моделей (группа 05) |
| Нет fine-tuning | Partial fine-tuning Whisper encoder (группа 04) |
| Нет централизованного логирования | MLflow: метрики, параметры, артефакты, кривые обучения |

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

exp_root = Path().resolve()

## 1. Сбор result.csv из всех экспериментов

In [ ]:
GROUP_COLORS = {
    "01_baselines":           "#4e9af1",
    "02_spectrogram_models":  "#f1a44e",
    "03_pretrained_frozen":   "#5cb85c",
    "04_pretrained_finetuned": "#9b59b6",
    "05_ensembles":           "#e74c3c",
    "06_phoneme_analysis":    "#95a5a6",
}

rows = []
for result_csv in sorted(exp_root.glob("**/result.csv")):
    group = result_csv.parts[-3] if len(result_csv.parts) >= 3 else "unknown"
    try:
        df = pd.read_csv(result_csv)
        df["group"] = group
        rows.append(df)
    except Exception as e:
        print(f"Ошибка чтения {result_csv}: {e}")

if rows:
    summary = pd.concat(rows, ignore_index=True)
    summary.to_csv(exp_root / "metrics_summary.csv", index=False, encoding="utf-8")
    print(f"Собрано {len(summary)} результатов")
    display(
        summary[
            ["experiment_id", "experiment_name", "group",
             "accuracy", "f1_macro", "f1_bad", "roc_auc",
             "threshold", "cv_f1_bad_std"]
        ].sort_values("f1_bad", ascending=False)
    )
else:
    print("Нет result.csv — запустите эксперименты")

## 2. График: F1-bad по всем экспериментам

In [ ]:
if len(rows) == 0:
    print("Нет данных для графика")
else:
    df_plot = summary.dropna(subset=["f1_bad"]).sort_values("f1_bad", ascending=True)
    colors = [GROUP_COLORS.get(g, "grey") for g in df_plot["group"]]

    fig, ax = plt.subplots(figsize=(10, max(6, len(df_plot) * 0.4)))
    bars = ax.barh(df_plot["experiment_id"], df_plot["f1_bad"], color=colors)

    if "cv_f1_bad_std" in df_plot.columns:
        std_vals = df_plot["cv_f1_bad_std"].fillna(0)
        ax.errorbar(
            df_plot["f1_bad"], range(len(df_plot)),
            xerr=std_vals, fmt="none", ecolor="black",
            elinewidth=1.5, capsize=4
        )

    for bar, v in zip(bars, df_plot["f1_bad"]):
        ax.text(v + 0.004, bar.get_y() + bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=8)

    ax.set_xlabel("F1-bad", fontsize=12)
    ax.set_title("F1-bad по экспериментам (holdout test)", fontsize=13)
    ax.set_xlim(0, 1.0)

    patches = [mpatches.Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, loc="lower right", fontsize=8)

    plt.tight_layout()
    plt.savefig(exp_root / "f1_bad_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("График сохранён в f1_bad_comparison.png")

## 3. Сравнение по группам (box plot)

In [ ]:
if len(rows) > 0 and len(summary) >= 4:
    group_order = list(GROUP_COLORS.keys())
    group_data = [
        summary[summary["group"] == g]["f1_bad"].dropna().values
        for g in group_order
        if g in summary["group"].values
    ]
    group_labels = [g for g in group_order if g in summary["group"].values]
    group_labels_short = [g.split("_", 1)[1] if "_" in g else g for g in group_labels]

    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(group_data, labels=group_labels_short, patch_artist=True)
    for patch, g in zip(bp["boxes"], group_labels):
        patch.set_facecolor(GROUP_COLORS.get(g, "grey"))
    ax.set_ylabel("F1-bad")
    ax.set_title("Распределение F1-bad по группам экспериментов")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(exp_root / "f1_bad_by_group.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. Сравнение с checkpoint_3 (исторические результаты)

In [ ]:
checkpoint3_best = pd.DataFrame([
    # 01_baselines
    {"experiment": "exp_01 (MFCC SVM)",             "f1_bad": 0.568, "roc_auc": 0.752, "source": "checkpoint_3"},
    {"experiment": "exp_02 (VGGish SVM)",            "f1_bad": 0.593, "roc_auc": 0.769, "source": "checkpoint_3"},
    # 02_spectrogram_models
    {"experiment": "exp_10 (DSSCNet)",               "f1_bad": 0.624, "roc_auc": 0.797, "source": "checkpoint_3"},
    {"experiment": "exp_14 (BiLSTM-GRU)",            "f1_bad": 0.601, "roc_auc": 0.769, "source": "checkpoint_3"},
    {"experiment": "exp_12 (FluentNet)",             "f1_bad": 0.610, "roc_auc": 0.785, "source": "checkpoint_3"},
    {"experiment": "exp_15 (CrossAttn CNN-BiLSTM)",  "f1_bad": 0.602, "roc_auc": 0.767, "source": "checkpoint_3"},
    # 03_pretrained_frozen
    {"experiment": "exp_30 (Whisper-small SVM)",     "f1_bad": 0.722, "roc_auc": 0.890, "source": "checkpoint_3"},
    {"experiment": "exp_36 (Whisper-medium SVM)",    "f1_bad": 0.747, "roc_auc": 0.894, "source": "checkpoint_3"},
    {"experiment": "exp_37 (Whisper-large SVM)",     "f1_bad": 0.724, "roc_auc": 0.901, "source": "checkpoint_3"},
    {"experiment": "exp_38 (Whisper-dysarthria SVM)","f1_bad": 0.644, "roc_auc": 0.852, "source": "checkpoint_3"},
    {"experiment": "exp_11 (Wav2Vec2 DL)",           "f1_bad": 0.617, "roc_auc": 0.791, "source": "checkpoint_3"},
    {"experiment": "exp_23 (XLS-R SVM)",             "f1_bad": 0.642, "roc_auc": 0.833, "source": "checkpoint_3"},
    {"experiment": "exp_13 (HuBERT LSTM)",           "f1_bad": 0.628, "roc_auc": 0.823, "source": "checkpoint_3"},
    {"experiment": "exp_18 (OpenL3 SVM)",            "f1_bad": 0.664, "roc_auc": 0.817, "source": "checkpoint_3"},
])

if len(rows) > 0:
    experiments_new = summary[["experiment_id", "f1_bad", "roc_auc"]].copy()
    experiments_new.columns = ["experiment", "f1_bad", "roc_auc"]
    experiments_new["source"] = "experiments/ (исправлено)"
    combined = pd.concat([checkpoint3_best, experiments_new], ignore_index=True)
    display(combined.sort_values("f1_bad", ascending=False).reset_index(drop=True))
else:
    print("Данные experiments/ ещё не собраны")
    display(checkpoint3_best)

## 5. Ключевые выводы

In [ ]:
if len(rows) > 0:
    best_exp = summary.loc[summary["f1_bad"].idxmax()]
    print("=" * 60)
    print(f"Лучший эксперимент: {best_exp['experiment_id']}")
    print(f"  Модель:    {best_exp['experiment_name']}")
    print(f"  F1-bad:    {best_exp['f1_bad']:.4f}")
    print(f"  F1-macro:  {best_exp['f1_macro']:.4f}")
    print(f"  Accuracy:  {best_exp['accuracy']:.4f}")
    print(f"  ROC-AUC:   {best_exp['roc_auc']:.4f}")
    if not pd.isna(best_exp.get('cv_f1_bad_std')):
        print(f"  CV std:    ±{best_exp['cv_f1_bad_std']:.4f}")
    print("=" * 60)
else:
    print("Запустите эксперименты и перезапустите этот ноутбук.")

## 6. MLflow — просмотр всех запусков

Все эксперименты логируются в MLflow. Для запуска UI выполните в терминале:

```bash
cd experiments/
mlflow ui --backend-store-uri mlruns/
# Откройте http://127.0.0.1:5000 в браузере
```

В UI доступны:
- Таблица всех запусков с метриками (F1-bad, F1-macro, ROC-AUC)
- Кривые обучения по эпохам (для DL-моделей)
- Метрики по CV-фолдам (для SVM-экспериментов)
- Артефакты: графики training curves, phoneme_analysis.png
- Git commit и branch для каждого запуска (автоматически)

In [ ]:
# Программный доступ к MLflow — сводная таблица всех запусков
try:
    import mlflow
    sys.path.insert(0, str(exp_root))
    from shared import config

    mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
    client = mlflow.tracking.MlflowClient()

    all_runs = []
    for exp in client.search_experiments():
        runs = client.search_runs(
            experiment_ids=[exp.experiment_id],
            order_by=["metrics.test_f1_bad DESC"],
        )
        for run in runs:
            m = run.data.metrics
            p = run.data.params
            all_runs.append({
                "group":      exp.name,
                "run_name":   run.info.run_name,
                "f1_bad":     m.get("test_f1_bad"),
                "f1_macro":   m.get("test_f1_macro"),
                "roc_auc":    m.get("test_roc_auc"),
                "git_commit": run.data.tags.get("git_commit", "")[:8],
                "status":     run.info.status,
            })

    if all_runs:
        df_mlflow = pd.DataFrame(all_runs)
        display(
            df_mlflow.dropna(subset=["f1_bad"])
                     .sort_values("f1_bad", ascending=False)
                     .reset_index(drop=True)
        )
    else:
        print("MLflow runs не найдены — запустите эксперименты")

except Exception as e:
    print(f"MLflow недоступен или runs отсутствуют: {e}")